In [ ]:

import os
import time
import warnings
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")

# ============================================================
# Problem 3 — Crypto Blind Anomaly Hunt (precision-first update)
# ------------------------------------------------------------
# This version is intentionally stricter than the earlier notebook.
# Goal:
#   1) Use market files as minute-level baseline.
#   2) Score campaign-days instead of dumping every manager-linked trade.
#   3) Keep only the highest-confidence trade IDs per suspicious campaign.
#   4) Preserve exact required submission format and accepted taxonomy.
# ============================================================

DATA_DIR = "data_dir"
MARKET_DIR = os.path.join(DATA_DIR, "crypto-market")
TRADES_DIR = os.path.join(DATA_DIR, "crypto-trades")
OUTPUT_FILE = "submission.csv"

SYMBOLS = [
    "BTCUSDT", "ETHUSDT", "SOLUSDT", "XRPUSDT",
    "DOGEUSDT", "LTCUSDT", "BATUSDT", "USDCUSDT"
]

start_time = time.time()
print("=" * 78)
print("PROBLEM 3 — PRECISION-FOCUSED RUN")
print("=" * 78)


def safe_mean(series):
    vals = pd.to_numeric(series, errors="coerce").dropna()
    return float(vals.mean()) if len(vals) else 0.0


def rolling_zscore(series, window=240, min_periods=60):
    baseline = series.rolling(window, min_periods=min_periods).median()
    std = series.rolling(window, min_periods=min_periods).std().replace(0, np.nan)
    z = (series - baseline) / std
    return z.clip(-10, 10)


def load_symbol_data(symbol: str):
    market_path = os.path.join(MARKET_DIR, f"Binance_{symbol}_2026_minute.csv")
    trades_path = os.path.join(TRADES_DIR, f"{symbol}_trades.csv")

    market = pd.read_csv(market_path)
    trades = pd.read_csv(trades_path, parse_dates=["timestamp"])

    market["timestamp"] = pd.to_datetime(market["Date"])
    market = market.sort_values("timestamp").reset_index(drop=True)
    market["mid"] = (market["High"] + market["Low"]) / 2
    market["vol_z"] = rolling_zscore(market["Volume USDT"])
    market["tc_z"] = rolling_zscore(market["tradecount"])

    trades["minute"] = trades["timestamp"].dt.floor("min")
    trades = trades.merge(
        market[["timestamp", "Open", "High", "Low", "Close", "mid", "Volume USDT", "tradecount", "vol_z", "tc_z"]],
        left_on="minute",
        right_on="timestamp",
        how="left",
        suffixes=("", "_market")
    )

    trades["date"] = trades["timestamp"].dt.date.astype(str)
    trades["notional"] = trades["price"] * trades["quantity"]
    trades["dev_mid_bp"] = np.where(
        trades["mid"].notna() & (trades["mid"] != 0),
        (trades["price"].sub(trades["mid"]).abs() / trades["mid"]) * 10000,
        np.nan,
    )
    trades["close_dev_bp"] = np.where(
        trades["Close"].notna() & (trades["Close"] != 0),
        (trades["price"].sub(trades["Close"]).abs() / trades["Close"]) * 10000,
        np.nan,
    )
    return market, trades


def classify_campaign(symbol: str, campaign: pd.DataFrame, full_symbol_trades: pd.DataFrame):
    campaign = campaign.sort_values("timestamp").copy()
    n = len(campaign)
    trader_count = campaign["trader_id"].nunique()
    buy_count = int((campaign["side"] == "BUY").sum())
    sell_count = int((campaign["side"] == "SELL").sum())
    prices = campaign["price"].to_numpy()
    notionals = campaign["notional"].to_numpy()

    mean_notional = float(np.mean(notionals)) if len(notionals) else 0.0
    std_notional = float(np.std(notionals)) if len(notionals) else 0.0
    qty_cv = std_notional / mean_notional if mean_notional > 0 else np.nan
    price_range_bp = ((np.max(prices) - np.min(prices)) / np.mean(prices) * 10000) if np.mean(prices) else 0.0
    avg_vol_z = safe_mean(campaign["vol_z"])
    avg_tc_z = safe_mean(campaign["tc_z"])
    avg_dev_bp = safe_mean(campaign["dev_mid_bp"])

    monotonic_up = float((np.diff(prices) >= 0).mean()) if n > 1 else 1.0
    monotonic_down = float((np.diff(prices) <= 0).mean()) if n > 1 else 1.0
    monotonic_ratio = max(monotonic_up, monotonic_down)

    symbol_notional_median = float(full_symbol_trades["notional"].median())
    symbol_notional_p995 = float(full_symbol_trades["notional"].quantile(0.995))

    result = {
        "violation_type": None,
        "score": 0.0,
        "reason": "",
        "selected": pd.DataFrame(),
    }

    # 1) Definite USDC peg breaks
    if symbol == "USDCUSDT" and (campaign["price"].sub(1.0).abs() > 0.005).any():
        selected = campaign[campaign["price"].sub(1.0).abs() > 0.005].copy()
        score = 10 + float(selected["price"].sub(1.0).abs().max()) * 1000 + max(0, avg_tc_z)
        result.update({
            "violation_type": "peg_break",
            "score": score,
            "reason": "Stablecoin traded materially away from the $1.00 peg with supporting minute context.",
            "selected": selected,
        })
        return result

    # 2) Tight buy/sell reversals -> wash / round-trip wash
    if n >= 4 and buy_count > 0 and sell_count > 0 and trader_count <= 2 and qty_cv < 0.35 and price_range_bp < 50:
        selected = campaign.assign(_rank=(campaign["notional"] - campaign["notional"].median()).abs()) \
                           .sort_values("_rank") \
                           .head(min(6, n))
        score = 9 + n / 3 + max(0, avg_tc_z)
        result.update({
            "violation_type": "round_trip_wash" if trader_count > 1 else "wash_trading",
            "score": score,
            "reason": "Tight buy/sell reversal with little net ownership change and narrow price dispersion.",
            "selected": selected,
        })
        return result

    # 3) Repeated similar notionals -> structuring
    if n >= 6 and qty_cv < 0.10 and price_range_bp < 120:
        selected = campaign.assign(_rank=(campaign["notional"] - campaign["notional"].median()).abs()) \
                           .sort_values("_rank") \
                           .head(min(5, n))
        score = 9 + n / 3 + max(0, avg_vol_z)
        result.update({
            "violation_type": "coordinated_structuring" if trader_count > 1 else "aml_structuring",
            "score": score,
            "reason": "Repeated near-identical notionals in a tight band indicate a structuring-style pattern.",
            "selected": selected,
        })
        return result

    # 4) Same-direction sequence with price walk -> ramping / coordinated pump
    if n >= 6 and campaign["side"].nunique() == 1 and monotonic_ratio >= 0.65 and (avg_vol_z > 0.1 or avg_tc_z > 0.1 or avg_dev_bp > 40):
        selected = campaign.assign(_rank=campaign["dev_mid_bp"].fillna(0) + campaign["notional"].rank(pct=True) * 5) \
                           .sort_values("_rank", ascending=False) \
                           .head(min(5, n))
        score = 8 + n / 3 + max(0, avg_vol_z) + max(0, avg_tc_z) + avg_dev_bp / 100
        result.update({
            "violation_type": "coordinated_pump" if trader_count > 1 else "ramping",
            "score": score,
            "reason": "Same-direction sequence walks the market with supportive volume/tradecount context.",
            "selected": selected,
        })
        return result

    # 5) Multi-wallet first-leg burst -> placement smurfing
    if n >= 2 and trader_count > 1 and campaign["side"].nunique() == 1 and qty_cv < 0.20 and (avg_dev_bp > 35 or avg_tc_z > 1):
        score = 8 + n / 3 + max(0, avg_tc_z)
        result.update({
            "violation_type": "placement_smurfing",
            "score": score,
            "reason": "Multiple wallets entered together on one side with similar sizing.",
            "selected": campaign.copy(),
        })
        return result

    # 6) Single very large coordinated print -> manager consolidation
    if n == 1 and mean_notional >= symbol_notional_p995 and (
        float(campaign["dev_mid_bp"].fillna(0).iloc[0]) > 50 or float(campaign["tc_z"].fillna(0).iloc[0]) > 2
    ):
        score = 8 + (mean_notional / max(symbol_notional_median, 1.0))
        result.update({
            "violation_type": "manager_consolidation",
            "score": score,
            "reason": "Single outsized coordinated print stands out sharply from the symbol baseline.",
            "selected": campaign.copy(),
        })
        return result

    return result


submission_rows = []
log_rows = []

overall_detect_start = time.time()
for symbol in SYMBOLS:
    symbol_start = time.time()
    market, trades = load_symbol_data(symbol)

    raw_rows = len(trades)
    mgr_rows = int(trades["manager_id"].notna().sum())
    candidate_rows = 0

    # definite peg-break candidates from all USDC trades
    if symbol == "USDCUSDT":
        peg_breaks = trades[
            (trades["price"].sub(1.0).abs() > 0.005) &
            ((trades["quantity"] > 80) | (trades["tc_z"].fillna(0) > 0.5))
        ].copy()
        for _, r in peg_breaks.iterrows():
            candidate_rows += 1
            submission_rows.append({
                "symbol": symbol,
                "date": r["date"],
                "trade_id": r["trade_id"],
                "violation_type": "peg_break",
                "score": 10 + abs(float(r["price"]) - 1.0) * 1000,
                "remarks": f"USDC deviated to {r['price']:.6f}; qty={r['quantity']:.2f}; minute tc_z={float(r['tc_z']) if pd.notna(r['tc_z']) else 0.0:.2f}",
            })

    manager_trades = trades[trades["manager_id"].notna()].copy()
    accepted_campaigns = 0

    for (manager_id, trade_date), campaign in manager_trades.groupby(["manager_id", "date"]):
        out = classify_campaign(symbol, campaign, trades)
        if out["violation_type"] is None or out["score"] < 9:
            continue

        accepted_campaigns += 1
        selected = out["selected"].copy()
        selected["campaign_score"] = out["score"]
        selected["campaign_type"] = out["violation_type"]
        selected["campaign_reason"] = out["reason"]

        for _, r in selected.iterrows():
            candidate_rows += 1
            submission_rows.append({
                "symbol": symbol,
                "date": r["date"],
                "trade_id": r["trade_id"],
                "violation_type": out["violation_type"],
                "score": out["score"],
                "remarks": (
                    f"{out['reason']} manager={manager_id}; day={trade_date}; "
                    f"price={r['price']:.6f}; qty={r['quantity']:.6f}; "
                    f"vol_z={float(r['vol_z']) if pd.notna(r['vol_z']) else 0.0:.2f}; "
                    f"tc_z={float(r['tc_z']) if pd.notna(r['tc_z']) else 0.0:.2f}"
                ),
            })

    symbol_runtime = time.time() - symbol_start
    print(
        f"[SYMBOL] {symbol:<8} trades={raw_rows:<5} manager_rows={mgr_rows:<4} "
        f"candidate_rows={candidate_rows:<3} accepted_campaigns={accepted_campaigns:<2} runtime={symbol_runtime:.2f}s"
    )
    log_rows.append({
        "symbol": symbol,
        "trade_rows": raw_rows,
        "manager_rows": mgr_rows,
        "candidate_rows": candidate_rows,
        "accepted_campaigns": accepted_campaigns,
        "runtime_sec": round(symbol_runtime, 3),
    })

detect_runtime = time.time() - overall_detect_start
print("-" * 78)
print(f"[DETECT] raw candidate rows collected: {len(submission_rows)}")

submission = pd.DataFrame(submission_rows)
if submission.empty:
    raise ValueError("No candidates were generated. Check data paths and detection thresholds.")

submission = submission.sort_values(["symbol", "score"], ascending=[True, False]).drop_duplicates("trade_id")

# Precision cap: keep all hard one-off signals, then only top rows per symbol.
final_parts = []
for symbol, group in submission.groupby("symbol", sort=False):
    hard_keep = group[group["violation_type"].isin(["peg_break", "manager_consolidation"])]
    rest = group[~group.index.isin(hard_keep.index)].head(12)
    final_parts.append(pd.concat([hard_keep, rest], ignore_index=True))

final_submission = pd.concat(final_parts, ignore_index=True).drop_duplicates("trade_id")
final_submission = final_submission.sort_values(["symbol", "score"], ascending=[True, False]).reset_index(drop=True)

final_submission[["symbol", "date", "trade_id", "violation_type", "remarks"]].to_csv(OUTPUT_FILE, index=False)

total_runtime = time.time() - start_time
print(f"[FINAL] detection runtime: {detect_runtime:.2f} sec")
print(f"[FINAL] total runtime:     {total_runtime:.2f} sec")
print(f"[FINAL] final rows:        {len(final_submission)}")
print(f"[FINAL] output file:       {OUTPUT_FILE}")
print("=" * 78)

log_df = pd.DataFrame(log_rows)
print("\nPER-SYMBOL SUMMARY")
display(log_df)

print("\nFINAL SUBMISSION PREVIEW")
display(final_submission[["symbol", "date", "trade_id", "violation_type"]].head(20))

print("\nVIOLATION TYPE COUNTS")
display(final_submission["violation_type"].value_counts().rename_axis("violation_type").reset_index(name="count"))

print("\nEXAMPLE REMARKS")
display(final_submission[["symbol", "trade_id", "violation_type", "remarks"]].head(8))


PROBLEM 3 — PRECISION-FOCUSED RUN
[SYMBOL] BTCUSDT  trades=5045  manager_rows=45   candidate_rows=21  accepted_campaigns=4  runtime=0.90s
[SYMBOL] ETHUSDT  trades=4031  manager_rows=31   candidate_rows=16  accepted_campaigns=3  runtime=0.84s
[SYMBOL] SOLUSDT  trades=2529  manager_rows=29   candidate_rows=7   accepted_campaigns=2  runtime=0.68s
[SYMBOL] XRPUSDT  trades=2535  manager_rows=35   candidate_rows=27  accepted_campaigns=6  runtime=0.68s
[SYMBOL] DOGEUSDT trades=2033  manager_rows=33   candidate_rows=16  accepted_campaigns=3  runtime=0.69s
[SYMBOL] LTCUSDT  trades=1527  manager_rows=27   candidate_rows=14  accepted_campaigns=3  runtime=0.64s
[SYMBOL] BATUSDT  trades=535   manager_rows=35   candidate_rows=21  accepted_campaigns=4  runtime=0.64s
[SYMBOL] USDCUSDT trades=1019  manager_rows=19   candidate_rows=11  accepted_campaigns=4  runtime=0.67s
------------------------------------------------------------------------------
[DETECT] raw candidate rows collected: 133
[FINAL] dete

,symbol,trade_rows,manager_rows,candidate_rows,accepted_campaigns,runtime_sec
0,BTCUSDT,5045,45,21,4,0.895
1,ETHUSDT,4031,31,16,3,0.842
2,SOLUSDT,2529,29,7,2,0.678
3,XRPUSDT,2535,35,27,6,0.682
4,DOGEUSDT,2033,33,16,3,0.692
5,LTCUSDT,1527,27,14,3,0.641
6,BATUSDT,535,35,21,4,0.638
7,USDCUSDT,1019,19,11,4,0.670



FINAL SUBMISSION PREVIEW


,symbol,date,trade_id,violation_type
0,BATUSDT,2026-01-28,BATUSDT_00000510,wash_trading
1,BATUSDT,2026-01-28,BATUSDT_00000513,wash_trading
2,BATUSDT,2026-01-28,BATUSDT_00000512,wash_trading
3,BATUSDT,2026-01-28,BATUSDT_00000511,wash_trading
4,BATUSDT,2026-01-28,BATUSDT_00000518,wash_trading
5,BATUSDT,2026-01-28,BATUSDT_00000514,wash_trading
6,BATUSDT,2026-02-23,BATUSDT_00000533,coordinated_structuring
7,BATUSDT,2026-02-23,BATUSDT_00000529,coordinated_structuring
8,BATUSDT,2026-02-23,BATUSDT_00000530,coordinated_structuring
9,BATUSDT,2026-02-23,BATUSDT_00000531,coordinated_structuring



VIOLATION TYPE COUNTS


,violation_type,count
0,aml_structuring,27
1,coordinated_structuring,20
2,round_trip_wash,14
3,wash_trading,12
4,ramping,7
5,placement_smurfing,4
6,peg_break,3
7,manager_consolidation,1



EXAMPLE REMARKS


,symbol,trade_id,violation_type,remarks
0,BATUSDT,BATUSDT_00000510,wash_trading,Tight buy/sell reversal with little net owners...
1,BATUSDT,BATUSDT_00000513,wash_trading,Tight buy/sell reversal with little net owners...
2,BATUSDT,BATUSDT_00000512,wash_trading,Tight buy/sell reversal with little net owners...
3,BATUSDT,BATUSDT_00000511,wash_trading,Tight buy/sell reversal with little net owners...
4,BATUSDT,BATUSDT_00000518,wash_trading,Tight buy/sell reversal with little net owners...
5,BATUSDT,BATUSDT_00000514,wash_trading,Tight buy/sell reversal with little net owners...
6,BATUSDT,BATUSDT_00000533,coordinated_structuring,Repeated near-identical notionals in a tight b...
7,BATUSDT,BATUSDT_00000529,coordinated_structuring,Repeated near-identical notionals in a tight b...
